> Federated Learning workflow with Autonomous client and model aggregation methos selection

### 1. Import Libraries

In [ ]:
from collections import deque
import copy
from dataclasses import dataclass, asdict
import json
import logging
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchmetrics import Accuracy, Precision, Recall
import torchvision
from torchvision import datasets, transforms
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple
import time
import random
import seaborn as sns
import syft as sy

import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

training_harware = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using training_harware: {training_harware}")

### 2. Model building

In [ ]:
class SimpleNet(nn.Module): # A simple CNN for image classification
   def __init__(self, in_channels: int = 1, num_classes: int = 10):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(SimpleNet, self).__init__()

       # 1st convolutional layer
       self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
       # Max pooling layer
       self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
       # 2nd convolutional layer
       self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
       # Fully connected layer
       self.fc1 = nn.Linear(16 * 7 * 7, num_classes)

   def forward(self, x):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
       x = self.pool(x)           # Apply max pooling
       x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
       x = self.pool(x)           # Apply max pooling
       x = x.reshape(x.shape[0], -1)  # Flatten the tensor
       x = self.fc1(x)            # Apply fully connected layer
       return x

In [ ]:
def local_train(model:SimpleNet, loader, epochs: int = 1, lr: float = 0.01, resources_info: Dict = {}) -> Tuple[float, Dict]:
    model.train()
    # Define the loss function
    criterion = nn.CrossEntropyLoss()

    # Define the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        for batch_index, (data, targets) in enumerate(tqdm(loader, desc="Training", leave=False)):
            data = data.to(training_harware)
            targets = targets.to(training_harware)
            output = model(data)
            loss = criterion(output, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    training_time = random.uniform(0.5, 2.0) * epochs  # Simulated training time
    
    return training_time, model.state_dict()

def evaluate(model, loader):
    # Set up of multiclass accuracy metric
    acc = Accuracy(task="multiclass",num_classes=10)

    # Iterate over the dataset batches
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            # Get predicted probabilities for test data batch
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            acc(preds, labels)
            # precision = Precision(preds, labels)
            # recall = Recall(preds, labels)

    #Compute total test accuracy
    test_accuracy = acc.compute()
    print(f"Test accuracy: {test_accuracy}")

    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
    accuracy = correct / len(loader.dataset)
    print(f"✅ Global model accuracy: {accuracy:.4f}")
    return accuracy


### 3. Dataset preparation

In [ ]:
batch_size = 60

train_dataset = datasets.MNIST(root="data/", download=True, train=True, transform=transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = datasets.MNIST(root="data/", download=True, train=False, transform=transforms.ToTensor())

test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


### 4. IoT devices/clients building

In [ ]:
# IoT Device and Gateway Communication Simulation
@dataclass
class BatteryState:
    design_capacity: float  # mAh
    current_level: float  # percentage (0-100)
    voltage: float       # volts
    discharge_rate: float # mA/h
    estimated_remaining: float # hours
    temperature: float   # Celsius (affects battery performance)
    actual_capacity: float = 0.0 # mAh

@dataclass
class BandwidthState:
    uplink_capacity: float    # Mbps
    uplink_usage: float       # Mbps
    downlink_capacity: float  # Mbps
    downlink_usage: float     # Mbps
    packet_loss: float        # percentage
    latency: float           # milliseconds
    jitter: float            # milliseconds

@dataclass
class MemoryState:
    total_ram: int          # KB
    used_ram: int           # KB
    free_ram: int           # KB
    buffer_usage: int       # KB
    fragmentation: float    # percentage

@dataclass
class Packet:
    source_id: str
    destination: str
    size: int              # bytes
    timestamp: float
    packet_type: str
    # priority: int          # 1-5 (5 = highest priority)
    retry_count: int       # number of retransmission attempts
    data: Optional[Dict] = None


In [ ]:
class IoTDevice:
    def __init__(
        self,
        device_id: str,
        device_type: str = "sensor",
        initial_battery: float = 100.0,
        location: Tuple[float, float] = (0.0, 0.0),
        local_model: SimpleNet = SimpleNet(),
        train_loader: DataLoader = None,
    ):
        self.device_id = device_id
        self.device_type = device_type
        self.location = location  # (x, y) coordinates
        self.creation_time = time.time()

        # ML model for federated learning
        self.local_model = local_model
        self.train_loader = train_loader

        # Real-world IoT device specifications
        device_specs = self._get_device_specifications(device_type)

        self.battery = BatteryState(
            design_capacity=device_specs["battery_capacity"],
            actual_capacity=device_specs[
                "battery_capacity"
            ],  # Initial capacity matches design
            current_level=initial_battery,
            voltage=device_specs["battery_voltage"],
            discharge_rate=device_specs["discharge_rate"],  # Real mAh consumption
            estimated_remaining=0.0,
            temperature=random.uniform(20.0, 35.0),  # Operating temperature
        )

        # Real-world IoT connectivity (varies with network conditions)
        self.bandwidth = BandwidthState(
            uplink_capacity=device_specs["max_uplink"],
            uplink_usage=0.0,
            downlink_capacity=device_specs["max_downlink"],
            downlink_usage=0.0,
            packet_loss=0.0,
            latency=random.uniform(10, 50),  # Base latency in ms
            jitter=random.uniform(1, 5),  # Base jitter in ms
        )

        # Real-world IoT memory constraints
        self.memory = MemoryState(
            total_ram=device_specs["total_ram"],
            used_ram=0,
            free_ram=0,
            buffer_usage=0,
            fragmentation=0.0,
        )

        self.is_active = True
        self.packet_queue = deque(maxlen=device_specs["queue_limit"])
        self.last_transmission = time.time()
        self.network_condition = "GOOD"  # Current network state
        self.energy_harvesting = device_specs.get("energy_harvesting", False)
        self.sleep_mode = False

        # Update initial memory state
        self.stats = {
            "packets_sent": 0,
            "packets_failed": 0,
            "total_data_sent": 0,
            "avg_power_consumption": 0.0,
            "connection_uptime": 0.0,
        }

        self._update_memory_state()
        logger.info(
            f"Device {device_id} ({device_type}) initialized at {location} with {self.battery.design_capacity}mAh battery"
        )

    def _get_device_specifications(self, device_type: str) -> Dict:
        """Get realistic specifications for different IoT device types"""
        specs = {
            "sensor": {
                "total_ram": random.randint(8, 64),
                "battery_voltage": 3.3,
                "battery_capacity": random.uniform(500, 2000),
                "discharge_rate": random.uniform(0.5, 5.0),  # mAh/h
                "max_uplink": 0.025,  # LoRaWAN typical
                "max_downlink": 0.025,
                "queue_limit": 10,
                "energy_harvesting": random.choice(
                    [True, False]
                ),  # Some sensors have solar panels
            },
            "actuator": {
                "total_ram": random.randint(16, 128),
                "battery_voltage": 3.7,
                "battery_capacity": random.uniform(1000, 3000),
                "discharge_rate": random.uniform(5.0, 20.0),
                "max_uplink": 0.1,
                "max_downlink": 0.2,
                "queue_limit": 20,
                "energy_harvesting": False,
            },
            "camera": {
                "total_ram": random.randint(512, 2048),
                "battery_voltage": 3.7,
                "battery_capacity": random.uniform(3000, 8000),
                "discharge_rate": random.uniform(50.0, 200.0),
                "max_uplink": 2.0,
                "max_downlink": 0.5,
                "queue_limit": 50,
                "energy_harvesting": False,
            },
            "gateway": {
                "total_ram": random.randint(1024, 8192),
                "battery_voltage": 5.0,
                "battery_capacity": random.uniform(10000, 20000),
                "discharge_rate": random.uniform(100.0, 500.0),
                "max_uplink": 10.0,
                "max_downlink": 10.0,
                "queue_limit": 100,
                "energy_harvesting": False,
            },
            "tracker": {
                "total_ram": random.randint(32, 256),
                "battery_voltage": 3.6,
                "battery_capacity": random.uniform(800, 2500),
                "discharge_rate": random.uniform(2.0, 15.0),
                "max_uplink": 0.05,
                "max_downlink": 0.05,
                "queue_limit": 15,
                "energy_harvesting": random.choice([True, False]),
            },
        }
        return specs.get(device_type, specs["sensor"])

    def can_participate(self) -> bool:
        """Check if device can participate in training"""
        if not self.is_active:
            return False
        
        # Check resource constraints
        battery_ok = self.battery.current_level > 10
        memory_ok = self.memory.free_ram > self.local_model.__sizeof__()
        network_ok = self.bandwidth.packet_loss < 10
        
        return battery_ok and memory_ok and network_ok
    
    def train_model_and_generate_packet(
        self,
        destination: str,
        epochs: int,
        round_num: int,
        global_update: dict[str, any],
    ) -> Packet:
        """Enhanced packet generation with priority and adaptive sizing"""
        packet_types = ["global_model", "local_update"]

        # Deive resource info
        resources_info = {
            "battery": asdict(self.battery),
            "bandwidth": asdict(self.bandwidth),
            "memory": asdict(self.memory),
            "location": self.location,
            "stats": self.stats.copy(),
        }

        # Load global updated model if available
        self.local_model.load_state_dict(global_update)
        # Local model training
        training_time, local_update = local_train(
            model=self.local_model,
            loader=self.train_loader,
            epochs=epochs,
            resources_info=resources_info,
        )
        packet_size = len(local_update) / (1024 * 1024)  # MB

        packet = Packet(
            source_id=self.device_id,
            destination=destination,
            size=packet_size,
            timestamp=time.time(),
            packet_type=packet_types[1],
            retry_count=0,
            data={
                'local_update': local_update,
                'training_time': training_time,
            },
        )

        if len(self.packet_queue) < self.packet_queue.maxlen:
            self.packet_queue.append(packet)
        else:
            logger.warning(f"Packet queue full for device {self.device_id}")

        return packet

    def transmit_packet(self) -> Optional[Tuple[bool, Packet]]:
        """Simulate packet transmission with resource updates"""
        if not self.packet_queue or self.battery.current_level < 5:
            return None

        # Enter sleep mode if battery is very low
        if self.battery.current_level < 10:
            self.sleep_mode = True
        else:
            self.sleep_mode = False

        packet = self.packet_queue.popleft()

        # Calculate transmission success probability
        success_probability = 0.95  # Base success rate
        if self.network_condition == "BAD":
            success_probability = 0.7
        if self.bandwidth.packet_loss > 10:
            success_probability *= 0.8

        # Simulate transmission attempt
        if random.random() > success_probability:
            packet.retry_count += 1
            if packet.retry_count < 3:  # Allow up to 3 retries
                self.packet_queue.appendleft(packet)  # Put back at front of queue
                self.stats["packets_failed"] += 1
                return None

        self.last_transmission = time.time()
        self.stats["packets_sent"] += 1
        self.stats["total_data_sent"] += packet.size

        return True, packet

    def get_status(self) -> Dict:
        """Get current device status"""
        uptime = time.time() - self.creation_time
        self.stats["connection_uptime"] = uptime

        return {
            "device_id": self.device_id,
            "device_type": self.device_type,
            "location": self.location,
            "is_active": self.is_active and self.battery.current_level > 0,
            "sleep_mode": self.sleep_mode,
            "battery": asdict(self.battery),
            "bandwidth": asdict(self.bandwidth),
            "memory": asdict(self.memory),
            "queue_size": len(self.packet_queue),
            "last_transmission": self.last_transmission,
            "network_condition": self.network_condition,
            "energy_harvesting": self.energy_harvesting,
            "stats": self.stats.copy(),
            "uptime_hours": uptime / 3600,
        }

### 5. Autonomic Computing building

In [ ]:
class ClientSelectionPolicy:
    """Client/Device selection policy framework"""

    @staticmethod
    def random_selection(
        devices: List[IoTDevice], selection_size: int
    ) -> List[IoTDevice]:
        """Random device selection"""
        eligible_devices = [d for d in devices if d.can_participate()]
        if len(eligible_devices) < selection_size:
            return eligible_devices
        return random.sample(eligible_devices, selection_size)

    @staticmethod
    def resource_aware_selection(
        devices: List[IoTDevice], selection_size: int
    ) -> List[IoTDevice]:
        """Resource-aware device selection (future implementation)"""
        eligible_devices = [d for d in devices if d.can_participate()]

        # Score devices based on resources
        scored_devices = []
        for device in eligible_devices:
            score = (
                device.battery_state.current_level * 0.4
                + (device.memory_state.free_ram / device.memory_state.total_ram)
                * 100
                * 0.3
                + (100 - device.bandwidth_state.packet_loss) * 0.3
            )
            scored_devices.append((device, score))

        # Select top devices
        scored_devices.sort(key=lambda x: x[1], reverse=True)
        return [d[0] for d in scored_devices[:selection_size]]

In [ ]:
class Gateway:
    def __init__(
        self,
        gateway_id: str = "gateway_001",
        location: Tuple[float, float] = (0.0, 0.0),
    ):
        self.gateway_id = gateway_id
        self.location = location
        self.connected_devices: Dict[str, IoTDevice] = {}
        self.selected_devices: List[IoTDevice] = []
        self.received_packets: List[Packet] = []
        self.monitoring_data: Dict[str, List[Dict]] = {}
        self.is_running = False
        self.creation_time = time.time()

        # Gateway capabilities
        self.total_bandwidth = 100.0  # Mbps
        self.current_bandwidth_usage = 0.0
        self.max_devices = 1000
        self.packet_processing_delay = 0.001  # seconds per packet

        # Gateway statistics
        self.stats = {
            "total_packets_processed": 0,
            "packets_dropped": 0,
            "average_latency": 0.0,
            "peak_bandwidth_usage": 0.0,
            "device_failures": 0,
        }
        self.packet_history = []
        self.round_statistics = []

        logger.info(f"Gateway {gateway_id} initialized at {location}")

    def register_device(self, device: IoTDevice):
        """Register an IoT device with the gateway"""
        if len(self.connected_devices) >= self.max_devices:
            logger.error(
                f"Gateway capacity exceeded. Cannot register device {device.device_id}"
            )
            return False

        self.connected_devices[device.device_id] = device
        self.monitoring_data[device.device_id] = []
        logger.info(
            f"Device {device.device_id} registered with gateway at {device.location}"
        )
        return True
    
    def _train_selection_model(self):
        """Train a decision tree model to predict device reliability"""
        pass

    def receive_packet(self, packet: Packet):
        """Process received packet and extract monitoring data"""
        processing_start = time.time()

        # Check if gateway is overloaded
        current_load = len(self.received_packets) * self.packet_processing_delay
        if current_load > 1.0:  # More than 1 second of processing backlog
            self.stats["packets_dropped"] += 1
            logger.warning(
                f"Packet dropped due to gateway overload from {packet.source_id}"
            )
            return False

        self.received_packets.append(packet)
        self.stats["total_packets_processed"] += 1

        # Store monitoring data
        if packet.data and packet.source_id in self.monitoring_data:
            data = {
                "timestamp": packet.timestamp,
                "packet_size": packet.size,
                "packet_type": packet.packet_type,
                # "priority": packet.priority,
                "retry_count": packet.retry_count,
                "processing_delay": time.time() - processing_start,
                **packet.data,
            }
            self.monitoring_data[packet.source_id].append(data)

            # Keep only last 1000 records per device to manage memory
            if len(self.monitoring_data[packet.source_id]) > 1000:
                self.monitoring_data[packet.source_id] = self.monitoring_data[
                    packet.source_id
                ][-1000:]

        # Update gateway statistics
        processing_time = time.time() - processing_start
        self.stats["average_latency"] = (
            self.stats["average_latency"] * 0.9 + processing_time * 0.1
        )

        return True

    def select_devices(
        self, devices: List[IoTDevice], selection_size: int, policy: str = "random"
    ) -> None:
        """Select devices for training round"""
        if policy == "random":
            self.selected_devices = ClientSelectionPolicy.random_selection(
                devices, selection_size
            )
        elif policy == "resource_aware":
            self.selected_devices = ClientSelectionPolicy.resource_aware_selection(
                devices, selection_size
            )
        else:
            raise ValueError(f"Unknown policy: {policy}")

    def broadcast_global_model(
        self,
        round_num: int,
        global_model: dict[str, any],
    ):
        """Simulate broadcasting global model to selected devices"""

        for device in self.selected_devices:
            packet = device.train_model_and_generate_packet(
                destination=self.gateway_id,
                epochs=1,
                global_update=global_model,
                round_num=round_num,
            )
            if packet:
                success, packet  = device.transmit_packet()
                if success and packet and packet.data:
                    self.receive_packet(packet)

        # This is a simplified simulation - just record the action
        print(
            f"Round {round_num}: Broadcasting global model to {len(self.selected_devices)} devices"
        )


### 6. Cloud Server building

In [ ]:
class AggregationMethods:
    """Federated learning aggregation methods"""

    def fed_avg(weights_list: List[Dict]) -> Dict:
        avg_weights = copy.deepcopy(weights_list[0])
        for key in avg_weights:
            for i in range(1, len(weights_list)):
                avg_weights[key] += weights_list[i][key]
            avg_weights[key] = avg_weights[key] / len(weights_list)
        return avg_weights

    def fed_median(self, weights_list):
        median_weights = copy.deepcopy(weights_list[0])
        for key in median_weights:
            all_weights = torch.stack([weights[key] for weights in weights_list], dim=0)
            median_weights[key] = torch.median(all_weights, dim=0).values
        return median_weights
    def fed_round_robin(self, weights_list, round_num):
        index = round_num % len(weights_list)
        return weights_list[index]
    def fed_weighted_avg(self, weights_list, weights):
        avg_weights = copy.deepcopy(weights_list[0])
        total_weight = sum(weights)
        for key in avg_weights:
            avg_weights[key] = sum(weights[i] * weights_list[i][key] for i in range(len(weights_list))) / total_weight
        return avg_weights
    def fed_krum(self, weights_list, f=1):
        num_clients = len(weights_list)
        scores = []
        for i in range(num_clients):
            distances = []
            for j in range(num_clients):
                if i != j:
                    dist = sum(torch.sum((weights_list[i][key] - weights_list[j][key])**2).item() for key in weights_list[i])
                    distances.append(dist)
            distances.sort()
            score = sum(distances[:num_clients - f - 2])
            scores.append(score)
        krum_index = scores.index(min(scores))
        return weights_list[krum_index]
    def fed_bulyan(self, weights_list, f=1):
        num_clients = len(weights_list)
        selected = []
        temp_weights = weights_list.copy()
        for _ in range(num_clients - 2 * f):
            scores = []
            for i in range(len(temp_weights)):
                distances = []
                for j in range(len(temp_weights)):
                    if i != j:
                        dist = sum(torch.sum((temp_weights[i][key] - temp_weights[j][key])**2).item() for key in temp_weights[i])
                        distances.append(dist)
                distances.sort()
                score = sum(distances[:num_clients - f - 2])
                scores.append(score)
            min_index = scores.index(min(scores))
            selected.append(temp_weights[min_index])
            temp_weights.pop(min_index)
        return self.fed_avg(selected)


In [ ]:
class CloudServer:
    def __init__(
        self, server_id: str = "cloud_server", global_model: SimpleNet = SimpleNet()
    ):
        self.server_id = server_id
        self.connected_gateways: Dict[str, Gateway] = {}
        self.global_model = global_model
        self.global_model_state = None
        self.creation_time = time.time()

        # Server capabilities
        self.model_update_interval = 3600  # 1 hour
        self.last_model_update = time.time()
        self.stats = {
            "total_gateways": 0,
            "total_devices": 0,
            "model_updates": 0,
            "aggregation_time": 0.0,
        }
        logger.info(f"Cloud server {server_id} initialized")

    def register_gateway(self, gateway: Gateway):
        """Register a gateway with the cloud server"""
        if gateway.gateway_id in self.connected_gateways:
            logger.warning(f"Gateway {gateway.gateway_id} already registered")
            return False

        self.connected_gateways[gateway.gateway_id] = gateway
        self.stats["total_gateways"] += len(gateway.connected_devices)
        logger.info(f"Gateway {gateway.gateway_id} registered with cloud server")
        return True

    def aggregate_model_updates(self, packets: List[Dict] = []):
        """Aggregate model updates from all connected clients/devices"""
        if not packets:
            logger.warning("No packets received for aggregation")
            return
        self.global_model_state = AggregationMethods.fed_avg(weights_list=packets)
        self.global_model.load_state_dict(self.global_model_state)
        self.last_model_update = time.time()
        logger.info(f"Global model updated with {len(packets)} local models")

    def distribute_global_model(self):
        """Distribute the global model to all connected gateways and their devices"""
        # for gateway in self.connected_gateways.values():
        #     for device in gateway.connected_devices.values():
        #         device.local_model.load_state_dict(self.global_model.state_dict())
        # logger.info("Global model distributed to all connected devices")
        return

### 7. Simulator building

In [ ]:
class FederatedLearningSystem:
    def __init__(
        self,
        selection_size: int = 6,
        simulation_area: Tuple[int, int] = (1000, 1000),
    ):
        self.num_devices: int = 0
        self.selection_size = selection_size
        self.gateway = Gateway()
        self.cloud_server = CloudServer()
        self.devices = List[IoTDevice]
        self.system_start_time = None
        self.system_end_time = None
        self.simulation_area = simulation_area  # (width, height) in meters
        self.real_time_data: Dict[str, Dict] = {}

        # Register gateway with Cloud Server
        self.cloud_server.register_gateway(self.gateway)

    # Create devices with configurations
    def add_device(
        self,
        device_id: str,
        device_type: str = "sensor",
        initial_battery: float = 100.0,
        location: Optional[Tuple[float, float]] = None,
    ):
        """Add IoT device with enhanced positioning and configuration"""
        if location is None:
            # Random placement within simulation area
            location = (
                random.uniform(0, self.simulation_area[0]),
                random.uniform(0, self.simulation_area[1]),
            )

        device = IoTDevice(
            device_id,
            device_type,
            initial_battery,
            location,
            local_model=SimpleNet(),
            train_loader=train_loader,
        )

        if self.gateway.register_device(device):
            self.num_devices += 1
            self.real_time_data[device_id] = {
                "timestamps": [],
                "battery": [],
                "memory": [],
                "bandwidth": [],
                "packet_loss": [],
                "latency": [],
                "temperature": [],
            }
            logger.info(f"Device {device_id} ({device_type}) added at {location}")
            return True
        return False

    def add_device_cluster(
        self,
        cluster_center: Tuple[float, float],
        cluster_radius: float,
        device_count: int,
        device_type: str = "sensor",
    ):
        """Add a cluster of devices around a central point"""
        for i in range(device_count):
            # Random position within cluster radius
            angle = random.uniform(0, 2 * np.pi)
            radius = random.uniform(0, cluster_radius)

            x = cluster_center[0] + radius * np.cos(angle)
            y = cluster_center[1] + radius * np.sin(angle)

            # Ensure device stays within simulation area
            x = max(0, min(x, self.simulation_area[0]))
            y = max(0, min(y, self.simulation_area[1]))

            device_id = f"{device_type}_cluster_{i:01d}"
            self.add_device(
                device_id,
                device_type,
                initial_battery=random.uniform(70, 100),
                location=(x, y),
            )

    def run_simulation(self, num_rounds: int = 10, policy: str = "random") -> Dict:
        """Run the complete federated learning simulation"""
        print(f"Starting Federated Learning Simulation")
        print(
            f"Devices: {self.num_devices}, Selection: {self.selection_size}, Rounds: {num_rounds}"
        )
        print(f"Policy: {policy}")
        print("=" * 60)

        self.system_start_time = time.time()

        for round_num in range(num_rounds):
            print(f"\n--- Round {round_num + 1}/{num_rounds} ---")

            # Record resource states before round
            # for device in self.gateway.connected_devices.values():
            #     device.record_resource_state(round_num)

            # Select devices for this round
            self.gateway.select_devices(
                self.gateway.connected_devices.values(), self.selection_size, policy
            )

            active_devices = len(
                [d for d in self.gateway.connected_devices.values() if d.is_active]
            )
            print(f"Active devices: {active_devices}/{self.num_devices}")
            print(f"Selected for training: {len(self.gateway.selected_devices)}")

            # Broadcast global model and Local Models train
            self.gateway.broadcast_global_model(
                round_num,
                self.cloud_server.global_model.state_dict(),
            )
            time.sleep(1)  # Simulate time delay for training

            # Aggregate updates at cloud server
            packets = [
                self.gateway.monitoring_data[dev.device_id][-1]["local_update"]
                for dev in self.gateway.selected_devices
                if self.gateway.monitoring_data[dev.device_id]
                and "local_update" in self.gateway.monitoring_data[dev.device_id][-1]
            ]

            print(f"Received {len(packets)} model updates for aggregation")
            self.cloud_server.aggregate_model_updates(packets=packets)
            self.cloud_server.distribute_global_model()

            time.sleep(1)  # Simulate time delay for distribution

            # Evaluate the global model for the current round
            eval_accuracy = evaluate(
                model=self.cloud_server.global_model, loader=test_loader
            )
            print(
                f"Global model evaluation accuracy | round {round_num}: {eval_accuracy*100:.2f}%"
            )

            # Generate round statistics

        # Evaluate the global model at the end
        eval_accuracy = evaluate(
            model=self.cloud_server.global_model, loader=test_loader
        )
        print(f"✅ Global model evaluation accuracy | final: {eval_accuracy*100:.2f}%")

        self.system_end_time = time.time()

        # Generate final system report

    def _generate_system_statistics(self) -> Dict:
        """Generate comprehensive system statistics"""
        total_time = self.system_end_time - self.system_start_time
        final_active = len(
            [d for d in self.gateway.connected_devices.values() if d.is_active]
        )

        # Calculate round statistics
        round_durations = [r["round_duration"] for r in self.gateway.round_statistics]
        successful_rates = [
            r["successful_updates"] / r["selected_devices"]
            for r in self.gateway.round_statistics
            if r["selected_devices"] > 0
        ]

        return {
            "total_execution_time": total_time,
            "final_active_devices": final_active,
            "avg_round_duration": np.mean(round_durations),
            "min_round_duration": np.min(round_durations),
            "max_round_duration": np.max(round_durations),
            "avg_success_rate": np.mean(successful_rates),
            "total_packets_sent": len(self.gateway.packet_history),
            "round_statistics": self.gateway.round_statistics,
        }

### 8. Run Simulation

In [ ]:
sim_env = FederatedLearningSystem(selection_size=6, simulation_area=(300, 300))

# Add various device clusters
sim_env.add_device_cluster(cluster_center=(50, 50), cluster_radius=30, device_count=10, device_type="sensor")
sim_env.add_device_cluster(cluster_center=(250, 250), cluster_radius=40, device_count=5, device_type="camera")
sim_env.add_device_cluster(cluster_center=(150, 150), cluster_radius=25, device_count=8, device_type="tracker")

# Add individual special devices
sim_env.add_device("temp_sensor", "sensor", initial_battery=95, location=(100, 200))
sim_env.add_device("security_camera", "camera", initial_battery=35, location=(20, 160))
sim_env.add_device("asset_tracker", "tracker", initial_battery=70, location=(130, 196))
sim_env.add_device("door_actuator", "actuator", initial_battery=60, location=(200, 100))

logger.info(f"Created simulation with {len(sim_env.gateway.connected_devices)} devices")

# Run simulation
sim_env.run_simulation(num_rounds=5, policy="random")
